# Librerias

In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from io import StringIO
import requests
import json 
import time 
import sqlite3
from urllib import robotparser

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

# Web Scraping

https://webscraper.io/test-sites/e-commerce/allinone/

Archivo robots.txt

https://webscraper.io/robots.txt

ser-agent: *
Esta regla es para todos. El asterisco es un comodín que incluye desde el buscador de Google hasta el script de python. No hay excepciones.

Disallow: /test-sites/e-commerce/ y /test-sites/tables 
Aquí el dueño del sitio está poniendo una "barrera". no extraer datos (No Scraping) de esas rutas específicas.

Sitemap: https://webscraper.io/sitemap.xml 
Es el "mapa del tesoro". Le dice a los bots dónde están todas las páginas importantes del sitio para que no tengan que adivinar. Es una cortesía del dueño para facilitar una extracción ordenada y eficiente.

In [18]:
dominio_base = "https://webscraper.io"
url_politicas = f"{dominio_base}/robots.txt"

print(f"Consultando reglas en: {url_politicas}")

response = requests.get(url_politicas, headers=headers)

if response.status_code == 200:
    print("--- Política de Robots Encontrada ---")
    print(response.text)
else:
    print("No se encontró el archivo robots.txt")

Consultando reglas en: https://webscraper.io/robots.txt
--- Política de Robots Encontrada ---
User-agent: *
Disallow: /test-sites/e-commerce/
Disallow: /test-sites/tables

Sitemap: https://webscraper.io/sitemap.xml



In [8]:
rp = robotparser.RobotFileParser()
rp.set_url(url_politicas)
rp.read()

user_agent = "*"
url_a_testear = "https://webscraper.io/test-sites/"
#url_a_testear = "https://webscraper.io/test-sites/e-commerce/allinone"

puede_entrar = rp.can_fetch(user_agent, url_a_testear)

if puede_entrar:
    print(f"✅ Acceso permitido a: {url_a_testear}")
else:
    print(f"❌ ACCESO DENEGADO por robots.txt en: {url_a_testear}")

✅ Acceso permitido a: https://webscraper.io/test-sites/


In [10]:
rp = robotparser.RobotFileParser()
rp.set_url(url_politicas)
rp.read()

user_agent = "*"
url_a_testear = "https://webscraper.io/test-sites/"

puede_entrar = rp.can_fetch(user_agent, url_a_testear)
delay = rp.crawl_delay(user_agent)

print(f"--- REPORTE DE GOBERNANZA ---")
if puede_entrar:
    print(f"✅ Permiso: Autorizado para la URL: {url_a_testear}")
    print(f"⏱️ Crawl-delay detectado: {delay if delay else 'No especificado (se recomienda 1s por cortesía)'}")
    
    try:
        response = requests.head(url_a_testear, timeout=10)
        
        print(f"\n--- REPORTE TÉCNICO DEL SERVIDOR ---")
        print(f"📡 Estado HTTP: {response.status_code}")
        
        content_type = response.headers.get('Content-Type', 'Desconocido')
        print(f"📄 Formato de datos: {content_type}")
        
        if 'text/html' in content_type:
            print("✔️ Validación: El recurso es apto para parseo con BeautifulSoup.")
        else:
            print("❌ Validación: El recurso NO es HTML. Revisar fuente antes de extraer.")

    except requests.exceptions.RequestException as e:
        print(f"❌ Error de conexión: {e}")

else:
    print(f"❌ ACCESO DENEGADO por robots.txt en: {url_a_testear}")

--- REPORTE DE GOBERNANZA ---
✅ Permiso: Autorizado para la URL: https://webscraper.io/test-sites/
⏱️ Crawl-delay detectado: No especificado (se recomienda 1s por cortesía)

--- REPORTE TÉCNICO DEL SERVIDOR ---
📡 Estado HTTP: 200
📄 Formato de datos: text/html; charset=UTF-8
✔️ Validación: El recurso es apto para parseo con BeautifulSoup.


### Web Scraping Basico

In [2]:
url_scraping = "https://webscraper.io/test-sites/e-commerce/allinone/"
datos_scraping = []

try:
    respuesta = requests.get(url_scraping, timeout=10)
    respuesta.raise_for_status() 
    
    sopa = BeautifulSoup(respuesta.text, 'html.parser')
    telefonos = sopa.find_all('div', class_='thumbnail')
    
    for tel in telefonos:
        try:
            nombre = tel.find('a', class_='title').text.strip()
            precio_str = tel.find('h4', class_='price').text.strip()
            reviews = tel.find('p', class_='review-count').text.strip()
            
            datos_scraping.append({
                'Nombre_Producto': nombre,
                'Precio_Crudo': precio_str,
                'Interacciones': reviews
            })
        except AttributeError:
            print("Se omitió un producto por falta de datos.")
            
    df_scraping = pd.DataFrame(datos_scraping)
    print(f"✅ Scraping completado: {len(df_scraping)} productos extraídos.")

except Exception as e:
    print(f"❌ Error crítico en el Web Scraping: {e}")

✅ Scraping completado: 3 productos extraídos.


In [3]:
df_scraping.head()

,Nombre_Producto,Precio_Crudo,Interacciones
0,Asus VivoBook...,$399.99,3 reviews
1,Ubuntu Edge,$499.99,2 reviews
2,Dell Inspiron...,$679,7 reviews


In [4]:
df_scraping.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Nombre_Producto  3 non-null      object
 1   Precio_Crudo     3 non-null      object
 2   Interacciones    3 non-null      object
dtypes: object(3)
memory usage: 204.0+ bytes


### Web Scraping Completo

In [5]:
url_scraping = "https://webscraper.io/test-sites/e-commerce/allinone/"
datos_scraping = []

try:
    respuesta = requests.get(url_scraping, timeout=10)
    respuesta.raise_for_status() 
    
    sopa = BeautifulSoup(respuesta.text, 'html.parser')
    telefonos = sopa.find_all('div', class_='thumbnail')
    
    for tel in telefonos:
        try:
            nombre = tel.find('a', class_='title').text.strip()
            precio_str = tel.find('h4', class_='price').text.strip()
            
            p_reviews = tel.find('p', class_='review-count')
            reviews = p_reviews.text.strip() if p_reviews else "0 reviews"
            
            p_desc = tel.find('p', class_='description')
            descripcion = p_desc.text.strip() if p_desc else "Sin descripción"
            
            p_rating = tel.find('p', attrs={'data-rating': True})
            estrellas = p_rating['data-rating'] if p_rating else "0"
            
            datos_scraping.append({
                'Nombre_Producto': nombre,
                'Precio_Crudo': precio_str,
                'Interacciones': reviews,
                'Descripcion': descripcion,
                'Calificacion': estrellas
            })
            
        except AttributeError as e:
            print(f"Se omitió un producto por falta de datos base.")
            
    df_scraping = pd.DataFrame(datos_scraping)
    print(f"✅ Scraping completado: {len(df_scraping)} productos extraídos.")

except Exception as e:
    print(f"❌ Error crítico en el Web Scraping: {e}")

✅ Scraping completado: 3 productos extraídos.


In [6]:
df_scraping.head()

,Nombre_Producto,Precio_Crudo,Interacciones,Descripcion,Calificacion
0,Asus VivoBook...,$399.99,3 reviews,"Asus VivoBook E502NA-GO022T Dark Blue, 15.6"" HD, Pentium N4200 1.1GHz, 4GB, 128GB SSD, Windows 10 Home, En/Ru kbd",4
1,Ubuntu Edge,$499.99,2 reviews,Sapphire glass,1
2,Dell Inspiron...,$679,7 reviews,"Dell Inspiron 15 (5567) Fog Gray, 15.6"" FHD, Core i5-7200U, 8GB, 1TB, Radeon R7 M445 4GB, Linux",2


In [7]:
df_scraping.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Nombre_Producto  3 non-null      object
 1   Precio_Crudo     3 non-null      object
 2   Interacciones    3 non-null      object
 3   Descripcion      3 non-null      object
 4   Calificacion     3 non-null      object
dtypes: object(5)
memory usage: 252.0+ bytes


## El Mapa HTML para Web Scraping

| Etiqueta HTML | Nombre Técnico | ¿Qué guardan normalmente los desarrolladores ahí? | ¿Qué le pedimos a BeautifulSoup? |
| :--- | :--- | :--- | :--- |
| `<div>` | División (Contenedor) | Son las "cajas grandes". Envuelven todo un producto (foto + texto + precio). | `soup.find('div', class_='...')` |
| `<span>` | Contenedor en línea | Textos pequeñitos, iconos, etiquetas de descuento (ej. "¡Oferta!"). | `soup.find('span').text` |
| `<h1>` a `<h6>` | Headings (Títulos) | `<h1>` es el título principal de la página. `<h3>` o `<h4>` suelen ser los nombres de los productos. | `soup.find('h3').text` |
| `<p>` | Párrafo | Descripciones largas, reseñas, características técnicas. | `soup.find('p').text` |
| `<a>` | Anchor (Enlace) | Botones o textos que te llevan a otra página. **OJO:** Aquí nos suele interesar la URL, no el texto. | `etiqueta['href']` |
| `<img>` | Imagen | Fotos de los productos. Aquí tampoco nos sirve el texto, queremos la URL de la foto. | `etiqueta['src']` |
| `<ul>` y `<li>` | Listas | Las viñetas (bullets) con las especificaciones técnicas (RAM, Disco Duro). | `soup.find_all('li')` |
| `<table>`, `<tr>`, `<td>`| Tablas | Fichas técnicas cuadriculadas. | *(Tip: Enséñales que Pandas tiene `pd.read_html()` que extrae tablas completas automáticamente).* |

### Web Scraping con todos los componentes del HTML

In [8]:
#url_scraping = "https://webscraper.io/test-sites/e-commerce/allinone/"
url_scraping = "https://webscraper.io/test-sites/e-commerce/allinone/phones/touch"

datos_scraping = []

df_scraping = pd.DataFrame() 

try:
    respuesta = requests.get(url_scraping, timeout=10)
    respuesta.raise_for_status() 
    
    sopa = BeautifulSoup(respuesta.text, 'html.parser')
    
    print("--- Extrayendo Menú Lateral (<ul> y <li>) ---")
    menu_lateral = sopa.find('ul', id='side-menu')
    if menu_lateral:
        lista_categorias = [li.text.strip().replace('\n', '') for li in menu_lateral.find_all('li')]
        print(f"Categorías encontradas: {lista_categorias}\n")

    print("--- Buscando Tablas (<table>) ---")
    try:
        html_seguro = StringIO(respuesta.text)
        tablas = pd.read_html(html_seguro)
        print(f"Se encontraron {len(tablas)} tablas en esta página.\n")
    except Exception as e: 
        print(f"Aviso: Falló la búsqueda de tablas. Detalle: {e}\n")

    print("--- Iniciando extracción de productos ---")
    telefonos = sopa.find_all('div', class_='thumbnail')
    
    for tel in telefonos:
        try:
            nombre = tel.find('a', class_='title').text.strip()
            precio_str = tel.find('h4', class_='price').text.strip()
            
            p_reviews = tel.find('p', class_='review-count')
            reviews = p_reviews.text.strip() if p_reviews else "0 reviews"
            
            p_desc = tel.find('p', class_='description')
            descripcion = p_desc.text.strip() if p_desc else "Sin descripción"
            
            p_rating = tel.find('p', attrs={'data-rating': True})
            estrellas_attr = p_rating['data-rating'] if p_rating else "0"
            
            etiqueta_img = tel.find('img', class_='img-responsive')
            url_imagen = etiqueta_img['src'] if etiqueta_img else "Sin imagen"
            
            spans_estrellas = tel.find_all('span', class_='ws-icon-star')
            conteo_spans = len(spans_estrellas) 
            
            datos_scraping.append({
                'Nombre_Producto': nombre,
                'Precio_Crudo': precio_str,
                'Interacciones': reviews,
                'Descripcion': descripcion,
                'Rating_Atributo': estrellas_attr,
                'Rating_Por_Spans': conteo_spans,
                'URL_Imagen': url_imagen          
            })
            
        except AttributeError:
            print("Se omitió un producto por falta de datos base.")
            
    # Solo intentamos crear el DF si logramos sacar datos
    if datos_scraping:
        df_scraping = pd.DataFrame(datos_scraping)
        print(f"✅ Scraping completado: {len(df_scraping)} productos extraídos.")

except Exception as e:
    print(f"❌ Error crítico en el Web Scraping: {e}")

if not df_scraping.empty:
    display(df_scraping.head(3))
else:
    print("⚠️ No hay datos para mostrar en la tabla.")

--- Extrayendo Menú Lateral (<ul> y <li>) ---
Categorías encontradas: ['Home', 'Computers', 'PhonesTouch', 'Touch']

--- Buscando Tablas (<table>) ---
Aviso: Falló la búsqueda de tablas. Detalle: No tables found

--- Iniciando extracción de productos ---
✅ Scraping completado: 9 productos extraídos.


,Nombre_Producto,Precio_Crudo,Interacciones,Descripcion,Rating_Atributo,Rating_Por_Spans,URL_Imagen
0,Nokia 123,$24.99,11 reviews,7 day battery,3,3,/images/test-sites/e-commerce/items/cart2.png
1,LG Optimus,$57.99,11 reviews,"3.2"" screen",3,3,/images/test-sites/e-commerce/items/cart2.png
2,Samsung Galaxy,$93.99,3 reviews,5 mpx. Android 5.0,3,3,/images/test-sites/e-commerce/items/cart2.png


In [20]:
df_scraping.head()

,Nombre_Producto,Precio_Crudo,Interacciones,Descripcion,Rating_Atributo,Rating_Por_Spans,URL_Imagen
0,Nokia 123,$24.99,11 reviews,7 day battery,3,3,/images/test-sites/e-commerce/items/cart2.png
1,LG Optimus,$57.99,11 reviews,"3.2"" screen",3,3,/images/test-sites/e-commerce/items/cart2.png
2,Samsung Galaxy,$93.99,3 reviews,5 mpx. Android 5.0,3,3,/images/test-sites/e-commerce/items/cart2.png
3,Nokia X,$109.99,4 reviews,"Andoid, Jolla dualboot",4,4,/images/test-sites/e-commerce/items/cart2.png
4,Sony Xperia,$118.99,6 reviews,"GPS, waterproof",1,1,/images/test-sites/e-commerce/items/cart2.png


# API

Identificamos que llega en el json de respuesta

In [21]:
url_prueba = "https://dummyjson.com/products" 
respuesta = requests.get(url_prueba)
datos = respuesta.json()

In [22]:
datos

{'products': [{'id': 1,
   'title': 'Essence Mascara Lash Princess',
   'description': 'The Essence Mascara Lash Princess is a popular mascara known for its volumizing and lengthening effects. Achieve dramatic lashes with this long-lasting and cruelty-free formula.',
   'category': 'beauty',
   'price': 9.99,
   'discountPercentage': 10.48,
   'rating': 2.56,
   'stock': 99,
   'tags': ['beauty', 'mascara'],
   'brand': 'Essence',
   'sku': 'BEA-ESS-ESS-001',
   'weight': 4,
   'dimensions': {'width': 15.14, 'height': 13.08, 'depth': 22.99},
   'warrantyInformation': '1 week warranty',
   'shippingInformation': 'Ships in 3-5 business days',
   'availabilityStatus': 'In Stock',
   'reviews': [{'rating': 3,
     'comment': 'Would not recommend!',
     'date': '2025-04-30T09:41:02.053Z',
     'reviewerName': 'Eleanor Collins',
     'reviewerEmail': 'eleanor.collins@x.dummyjson.com'},
    {'rating': 4,
     'comment': 'Very satisfied!',
     'date': '2025-04-30T09:41:02.053Z',
     'review

In [23]:
print(f"Total de registros: {datos.get('total')} productos.")
print(f"Limite de registros: {datos.get('limit')} productos.")

Total de registros: 194 productos.
Limite de registros: 30 productos.


Empezamos a extraer informacion

In [24]:
url_api = "https://dummyjson.com/products/"
#url_api = "https://dummyjson.com/products?limit=30&skip=30"
#url_api = "https://dummyjson.com/products?limit=0"

datos_api = []

try:
    respuesta_api = requests.get(url_api, timeout=10)
    
    if respuesta_api.status_code == 200:
        json_data = respuesta_api.json()
        
        productos = json_data.get('products', [])
        
        for prod in productos:
            datos_api.append({
                'Titulo': prod.get('title'),
                'Precio_Dolares': prod.get('price'),
                'Rating_Stars': prod.get('rating')
            })
            
        df_api = pd.DataFrame(datos_api)
        print(f"✅ API completada: {len(df_api)} productos descargados.")
    else:
        print(f"⚠️ La API respondió con código: {respuesta_api.status_code}")

except Exception as e:
    print(f"❌ Error al conectar con la API: {e}")

✅ API completada: 30 productos descargados.


In [25]:
df_api.head()

,Titulo,Precio_Dolares,Rating_Stars
0,Essence Mascara Lash Princess,9.99,2.56
1,Eyeshadow Palette with Mirror,19.99,2.86
2,Powder Canister,14.99,4.64
3,Red Lipstick,12.99,4.36
4,Red Nail Polish,8.99,4.32


Extraccion por lotes

In [26]:
datos_api = []

url_inicial = "https://dummyjson.com/products/"
respuesta_inicial = requests.get(url_inicial).json()
total_productos = respuesta_inicial.get('total', 0)
limite_por_pagina = respuesta_inicial.get('limit', 0)

print(f"El proveedor tiene {total_productos} productos en su bodega.")
print(f"Iniciando descarga por lotes de {limite_por_pagina} en {limite_por_pagina}...\n")

for salto in range(0, total_productos, limite_por_pagina):

    url_paginada = f"https://dummyjson.com/products?limit={limite_por_pagina}&skip={salto}"
    
    try:
        respuesta = requests.get(url_paginada, timeout=10)
        
        if respuesta.status_code == 200:
            json_data = respuesta.json()
            productos = json_data.get('products', [])
            
            for prod in productos:
                datos_api.append({
                    'Titulo': prod.get('title'),
                    'Precio_Dolares': prod.get('price'),
                    'Categoria': prod.get('category', 'Sin categoría')
                })
            
            print(f"✅ Lote descargado. (Saltamos {salto} - Extraídos {len(productos)} nuevos)")
        else:
            print(f"⚠️ Error en el lote (Skip {salto}). Código: {respuesta.status_code}")
      
        time.sleep(1)

    except Exception as e:
        print(f"❌ Error al conectar en el salto {salto}: {e}")


df_api = pd.DataFrame(datos_api)

El proveedor tiene 194 productos en su bodega.
Iniciando descarga por lotes de 30 en 30...

✅ Lote descargado. (Saltamos 0 - Extraídos 30 nuevos)
✅ Lote descargado. (Saltamos 30 - Extraídos 30 nuevos)
✅ Lote descargado. (Saltamos 60 - Extraídos 30 nuevos)
✅ Lote descargado. (Saltamos 90 - Extraídos 30 nuevos)
✅ Lote descargado. (Saltamos 120 - Extraídos 30 nuevos)
✅ Lote descargado. (Saltamos 150 - Extraídos 30 nuevos)
✅ Lote descargado. (Saltamos 180 - Extraídos 14 nuevos)


In [27]:
df_api

,Titulo,Precio_Dolares,Categoria
0,Essence Mascara Lash Princess,9.99,beauty
1,Eyeshadow Palette with Mirror,19.99,beauty
2,Powder Canister,14.99,beauty
3,Red Lipstick,12.99,beauty
4,Red Nail Polish,8.99,beauty
5,Calvin Klein CK One,49.99,fragrances
6,Chanel Coco Noir Eau De,129.99,fragrances
7,Dior J'adore,89.99,fragrances
8,Dolce Shine Eau de,69.99,fragrances
9,Gucci Bloom Eau de,79.99,fragrances


Colocamos mas campos

In [28]:
url_api = "https://dummyjson.com/products?limit=0"
datos_api = []

try:
    respuesta_api = requests.get(url_api, timeout=10)
    
    if respuesta_api.status_code == 200:
        json_data = respuesta_api.json()
        
        productos = json_data.get('products', [])
        
        for prod in productos:
            datos_api.append({
                'Titulo': prod.get('title'),
                'Description': prod.get('description'),
                'Precio_Dolares': prod.get('price'),
                'Rating_Stars': prod.get('rating'),
                'Categoria': prod.get('category', 'Sin categoría'),
                'Marca': prod.get('brand', 'Generico'),
                'Stock_Disponible': prod.get('stock', 0),
                'Descuento_Pct': prod.get('discountPercentage', 0.0),
            })
            
        df_api = pd.DataFrame(datos_api)
        print(f"✅ API completada: {len(df_api)} productos descargados en total.")
    else:
        print(f"⚠️ La API respondió con código: {respuesta_api.status_code}")

except Exception as e:
    print(f"❌ Error al conectar con la API: {e}")

✅ API completada: 194 productos descargados en total.


In [29]:
df_api.head()

,Titulo,Description,Precio_Dolares,Rating_Stars,Categoria,Marca,Stock_Disponible,Descuento_Pct
0,Essence Mascara Lash Princess,The Essence Mascara Lash Princess is a popular mascara known for its volumizing and lengthening effects. Achieve dramatic lashes with this long-lasting and cruelty-free formula.,9.99,2.56,beauty,Essence,99,10.48
1,Eyeshadow Palette with Mirror,"The Eyeshadow Palette with Mirror offers a versatile range of eyeshadow shades for creating stunning eye looks. With a built-in mirror, it's convenient for on-the-go makeup application.",19.99,2.86,beauty,Glamour Beauty,34,18.19
2,Powder Canister,"The Powder Canister is a finely milled setting powder designed to set makeup and control shine. With a lightweight and translucent formula, it provides a smooth and matte finish.",14.99,4.64,beauty,Velvet Touch,89,9.84
3,Red Lipstick,"The Red Lipstick is a classic and bold choice for adding a pop of color to your lips. With a creamy and pigmented formula, it provides a vibrant and long-lasting finish.",12.99,4.36,beauty,Chic Cosmetics,91,12.16
4,Red Nail Polish,"The Red Nail Polish offers a rich and glossy red hue for vibrant and polished nails. With a quick-drying formula, it provides a salon-quality finish at home.",8.99,4.32,beauty,Nail Couture,79,11.44


Extraemos mas datos relevantes

### 🗂️ Anatomía de un JSON Anidado 

Cuando consumimos una API, los datos casi nunca vienen planos. Vienen en listas dentro de diccionarios, dentro de otras listas. 

Imagina que la variable `productos` es un **Cajón de Archivo**, cada producto es una **Carpeta**, y las reseñas son un **Sobre Blanco** escondido adentro de esa carpeta:

```javascript
[  // <-- EL CAJÓN PRINCIPAL (La lista total de productos)
   
   { // <-- Carpeta 1 (El diccionario del Producto 1)
      "id": 1,
      "title": "iPhone 9",
      "price": 549,
      
      "reviews": [ // <-- El sobre blanco (Una sub-lista de reseñas)
         { "rating": 5, "comment": "Excelente!" },         // Tarjetita 1
         { "rating": 3, "comment": "Batería regular." }    // Tarjetita 2
      ]
   },

   { // <-- Carpeta 2 (El diccionario del Producto 2)
      "id": 2,
      "title": "Samsung S20",
      "price": 899,
      
      "reviews": [ // <-- Otro sobre blanco
         { "rating": 4, "comment": "Muy rápido" }          // Tarjetita 1
      ]
   }

]

In [30]:
url_api = "https://dummyjson.com/products?limit=0"

datos_productos = []
datos_reviews = []

try:
    respuesta_api = requests.get(url_api, timeout=10)
    
    if respuesta_api.status_code == 200:
        json_data = respuesta_api.json()
        productos = json_data.get('products', [])
        
        for prod in productos:
            id_producto = prod.get('id')
            
            datos_productos.append({
                'ID_Producto': id_producto, 
                'Titulo': prod.get('title'),
                'Precio_Dolares': prod.get('price'),
                'Categoria': prod.get('category', 'Sin categoría'),
                'Marca': prod.get('brand', 'Generico')
            })
            
            lista_reviews = prod.get('reviews', [])
            
            for res in lista_reviews:
                datos_reviews.append({
                    'Producto_ID': id_producto, # <- El ancla de conexión
                    'Calificacion': res.get('rating'),
                    'Comentario': res.get('comment'),
                    'Date': res.get('date'),
                    'Usuario': res.get('reviewerName'),
                    'Fecha': res.get('date')
                })

        df_productos = pd.DataFrame(datos_productos)
        df_reviews = pd.DataFrame(datos_reviews)
        
        print("✅ Extracción completada con éxito.")
        print(f"Catálogo: {len(df_productos)} productos.")
        print(f"Reseñas totales: {len(df_reviews)} comentarios capturados.\n")
        
    else:
        print(f"⚠️ La API respondió con código: {respuesta_api.status_code}")

except Exception as e:
    print(f"❌ Error al conectar con la API: {e}")

✅ Extracción completada con éxito.
Catálogo: 194 productos.
Reseñas totales: 582 comentarios capturados.



In [31]:
df_reviews.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 582 entries, 0 to 581
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Producto_ID   582 non-null    int64 
 1   Calificacion  582 non-null    int64 
 2   Comentario    582 non-null    object
 3   Date          582 non-null    object
 4   Usuario       582 non-null    object
 5   Fecha         582 non-null    object
dtypes: int64(2), object(4)
memory usage: 27.4+ KB


Otros tipos de campos

In [56]:
import requests
import pandas as pd

url_api = "https://dummyjson.com/products?limit=0"

datos_productos = []
datos_reviews = []

try:
    respuesta_api = requests.get(url_api, timeout=10)
    
    if respuesta_api.status_code == 200:
        json_data = respuesta_api.json()
        productos = json_data.get('products', [])
        
        for prod in productos:
            id_producto = prod.get('id')
            
            lista_tags = prod.get('tags', [])
            tags_limpios = ", ".join(lista_tags) 
            
            dims = prod.get('dimensions', {})
            ancho = dims.get('width', 0)
            alto = dims.get('height', 0)
            profundidad = dims.get('depth', 0)

            datos_productos.append({
                'ID_Producto': id_producto,
                'Titulo': prod.get('title'),
                'Description': prod.get('description'),
                'Precio_Dolares': prod.get('price'),
                'Rating_Stars': prod.get('rating'),
                'Categoria': prod.get('category', 'Sin categoría'),
                'Marca': prod.get('brand', 'Generico'),
                'Stock_Disponible': prod.get('stock', 0),
                'Descuento_Pct': prod.get('discountPercentage', 0.0),
                'Etiquetas': tags_limpios,
                'Ancho_cm': ancho,
                'Alto_cm': alto,
                'Profundidad_cm': profundidad
            })
            
            lista_reviews = prod.get('reviews', [])
            
            for res in lista_reviews:
                datos_reviews.append({
                    'Producto_ID': id_producto, 
                    'Calificacion': res.get('rating'),
                    'Comentario': res.get('comment'),
                    'Fecha': res.get('date'),
                    'Usuario': res.get('reviewerName')
                })

        df_productos = pd.DataFrame(datos_productos)
        df_reviews = pd.DataFrame(datos_reviews)
        
        print("✅ Extracción Maestra completada con éxito.")
        print(f"Catálogo: {len(df_productos)} productos detallados.")
        print(f"Reseñas totales: {len(df_reviews)} comentarios capturados.\n")
        
    else:
        print(f"⚠️ La API respondió con código: {respuesta_api.status_code}")

except Exception as e:
    print(f"❌ Error al conectar con la API: {e}")

✅ Extracción Maestra completada con éxito.
Catálogo: 194 productos detallados.
Reseñas totales: 582 comentarios capturados.



In [57]:
df_productos.head()

,ID_Producto,Titulo,Description,Precio_Dolares,Rating_Stars,Categoria,Marca,Stock_Disponible,Descuento_Pct,Etiquetas,Ancho_cm,Alto_cm,Profundidad_cm
0,1,Essence Mascara Lash Princess,The Essence Mascara Lash Princess is a popular mascara known for its volumizing and lengthening effects. Achieve dramatic lashes with this long-lasting and cruelty-free formula.,9.99,2.56,beauty,Essence,99,10.48,"beauty, mascara",15.14,13.08,22.99
1,2,Eyeshadow Palette with Mirror,"The Eyeshadow Palette with Mirror offers a versatile range of eyeshadow shades for creating stunning eye looks. With a built-in mirror, it's convenient for on-the-go makeup application.",19.99,2.86,beauty,Glamour Beauty,34,18.19,"beauty, eyeshadow",9.26,22.47,27.67
2,3,Powder Canister,"The Powder Canister is a finely milled setting powder designed to set makeup and control shine. With a lightweight and translucent formula, it provides a smooth and matte finish.",14.99,4.64,beauty,Velvet Touch,89,9.84,"beauty, face powder",29.27,27.93,20.59
3,4,Red Lipstick,"The Red Lipstick is a classic and bold choice for adding a pop of color to your lips. With a creamy and pigmented formula, it provides a vibrant and long-lasting finish.",12.99,4.36,beauty,Chic Cosmetics,91,12.16,"beauty, lipstick",18.11,28.38,22.17
4,5,Red Nail Polish,"The Red Nail Polish offers a rich and glossy red hue for vibrant and polished nails. With a quick-drying formula, it provides a salon-quality finish at home.",8.99,4.32,beauty,Nail Couture,79,11.44,"beauty, nail polish",21.63,16.48,29.84


# Transformacion

Web Scraping

In [86]:
df_scraping_limpio = df_scraping.copy()

df_scraping_limpio['Precio'] = df_scraping_limpio['Precio_Crudo'].str.replace('$', '', regex=False).astype(float)

df_scraping_limpio['Total_Resenas'] = df_scraping_limpio['Interacciones'].str.replace(' reviews', '', regex=False).astype(int)

df_scraping_final = df_scraping_limpio[['Nombre_Producto', 'Precio', 'Rating_Atributo', 'Total_Resenas', 'Descripcion']].copy()

df_scraping_final.columns = ['Producto', 'Precio', 'Calificacion_Promedio', 'Total_Resenas', 'Descripcion']

df_scraping_final['Origen'] = 'WebScraping_TouchPhones'

In [87]:
df_scraping_final.head()

,Producto,Precio,Calificacion_Promedio,Total_Resenas,Descripcion,Origen
0,Nokia 123,24.99,3,11,7 day battery,WebScraping_TouchPhones
1,LG Optimus,57.99,3,11,"3.2"" screen",WebScraping_TouchPhones
2,Samsung Galaxy,93.99,3,3,5 mpx. Android 5.0,WebScraping_TouchPhones
3,Nokia X,109.99,4,4,"Andoid, Jolla dualboot",WebScraping_TouchPhones
4,Sony Xperia,118.99,1,6,"GPS, waterproof",WebScraping_TouchPhones


API

In [94]:
resumen_reviews = df_reviews.groupby('Producto_ID').agg(
    Promedio_API=('Calificacion', 'mean'), 
    Total_Resenas_API=('Calificacion', 'count') 
).reset_index() 

resumen_reviews['Promedio_API'] = resumen_reviews['Promedio_API'].round(1)

In [95]:
df_api_filtrado = df_productos[df_productos['Categoria'] == 'smartphones'].copy()

df_api_base = df_api_filtrado[['ID_Producto', 'Titulo', 'Precio_Dolares', 'Description']].copy()
df_api_base.columns = ['ID_Producto', 'Producto', 'Precio', 'Descripcion']

In [96]:
df_consolidado = pd.merge(df_api_base,resumen_reviews,left_on='ID_Producto', right_on='Producto_ID', how='left')

In [97]:
df_consolidado = df_consolidado.rename(columns={
    'Promedio_API': 'Calificacion_Promedio',
    'Total_Resenas_API': 'Total_Resenas'
})

df_consolidado = df_consolidado.drop(columns=['ID_Producto', 'Producto_ID'])
df_consolidado['Calificacion_Promedio'] = df_consolidado['Calificacion_Promedio'].fillna(0)
df_consolidado['Total_Resenas'] = df_consolidado['Total_Resenas'].fillna(0).astype(int)

df_consolidado['Origen'] = 'API_Smartphones'

In [98]:
df_consolidado.head()

,Producto,Precio,Descripcion,Calificacion_Promedio,Total_Resenas,Origen
0,iPhone 5s,199.99,"The iPhone 5s is a classic smartphone known for its compact design and advanced features during its release. While it's an older model, it still provides a reliable user experience.",3.7,3,API_Smartphones
1,iPhone 6,299.99,"The iPhone 6 is a stylish and capable smartphone with a larger display and improved performance. It introduced new features and design elements, making it a popular choice in its time.",4.0,3,API_Smartphones
2,iPhone 13 Pro,1099.99,"The iPhone 13 Pro is a cutting-edge smartphone with a powerful camera system, high-performance chip, and stunning display. It offers advanced features for users who demand top-notch technology.",4.3,3,API_Smartphones
3,iPhone X,899.99,"The iPhone X is a flagship smartphone featuring a bezel-less OLED display, facial recognition technology (Face ID), and impressive performance. It represents a milestone in iPhone design and innovation.",3.7,3,API_Smartphones
4,Oppo A57,249.99,"The Oppo A57 is a mid-range smartphone known for its sleek design and capable features. It offers a balance of performance and affordability, making it a popular choice.",3.7,3,API_Smartphones


In [99]:
df_consolidado.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16 entries, 0 to 15
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Producto               16 non-null     object 
 1   Precio                 16 non-null     float64
 2   Descripcion            16 non-null     object 
 3   Calificacion_Promedio  16 non-null     float64
 4   Total_Resenas          16 non-null     int64  
 5   Origen                 16 non-null     object 
dtypes: float64(2), int64(1), object(3)
memory usage: 900.0+ bytes


Union

In [130]:
df_master = pd.concat([df_scraping_final, df_consolidado], ignore_index=True)

In [119]:
df_master.head(10)

,Producto,Precio,Calificacion_Promedio,Total_Resenas,Descripcion,Origen
0,Nokia 123,24.99,3,11,7 day battery,WebScraping_TouchPhones
1,LG Optimus,57.99,3,11,"3.2"" screen",WebScraping_TouchPhones
2,Samsung Galaxy,93.99,3,3,5 mpx. Android 5.0,WebScraping_TouchPhones
3,Nokia X,109.99,4,4,"Andoid, Jolla dualboot",WebScraping_TouchPhones
4,Sony Xperia,118.99,1,6,"GPS, waterproof",WebScraping_TouchPhones
5,Ubuntu Edge,499.99,1,2,Sapphire glass,WebScraping_TouchPhones
6,Iphone,899.99,1,10,White,WebScraping_TouchPhones
7,Iphone,899.99,2,8,Silver,WebScraping_TouchPhones
8,Iphone,899.99,1,1,Black,WebScraping_TouchPhones
9,iPhone 5s,199.99,3.7,3,"The iPhone 5s is a classic smartphone known for its compact design and advanced features during its release. While it's an older model, it still provides a reliable user experience.",API_Smartphones


In [102]:
df_master['Origen'].value_counts()

Origen
API_Smartphones            16
WebScraping_TouchPhones     9
Name: count, dtype: int64

In [103]:
df_master.groupby('Origen')['Precio'].mean().round(2)

Origen
API_Smartphones            424.99
WebScraping_TouchPhones    400.66
Name: Precio, dtype: float64

In [104]:
df_master.sort_values(by='Precio', ascending=False).head(5)

,Producto,Precio,Calificacion_Promedio,Total_Resenas,Descripcion,Origen
11,iPhone 13 Pro,1099.99,4.3,3,"The iPhone 13 Pro is a cutting-edge smartphone with a powerful camera system, high-performance chip, and stunning display. It offers advanced features for users who demand top-notch technology.",API_Smartphones
6,Iphone,899.99,1,10,White,WebScraping_TouchPhones
8,Iphone,899.99,1,1,Black,WebScraping_TouchPhones
12,iPhone X,899.99,3.7,3,"The iPhone X is a flagship smartphone featuring a bezel-less OLED display, facial recognition technology (Face ID), and impressive performance. It represents a milestone in iPhone design and innovation.",API_Smartphones
7,Iphone,899.99,2,8,Silver,WebScraping_TouchPhones


# Carga

Uno a uno

In [105]:
url_crm = "https://webhook.site/644dc825-e1d0-4e49-a4f4-97f0aeba7a9d"

registros_a_enviar = df_master.to_dict(orient='records')

lotes_exitosos = 0
for i, registro in enumerate(registros_a_enviar, 1):
    try:
        respuesta = requests.post(url_crm, json=registro, timeout=5)
        
        if respuesta.status_code == 200:
            lotes_exitosos += 1
            if i <= 3:
                print(f"  -> ✅ Enviado: {registro['Producto'][:20]}...")
        else:
            print(f"  -> ⚠️ Error {respuesta.status_code} al enviar ID {i}")
            
    except requests.exceptions.RequestException as e:
        print(f"  -> ❌ Falla de conexión en registro {i}: {e}")
        
    time.sleep(0.5) 
    
print(f"\nSincronización Finalizada: {lotes_exitosos}/{len(registros_a_enviar)} registros integrados al CRM.")

  -> ✅ Enviado: Nokia 123...
  -> ✅ Enviado: LG Optimus...
  -> ✅ Enviado: Samsung Galaxy...

🚀 Sincronización Finalizada: 25/25 registros integrados al CRM.


Lotes

In [107]:
url_crm = "https://webhook.site/4fb63d87-165b-42df-9dee-2d17db06cb07"

registros_a_enviar = df_master.to_dict(orient='records')

print(f"Iniciando envío en bloque de {len(registros_a_enviar)} registros al CRM...")

try:
    respuesta = requests.post(url_crm, json=registros_a_enviar, timeout=15)
    
    if respuesta.status_code == 200:
        print(f"✅ ¡Sincronización Exitosa! El bloque completo fue recibido.")
    else:
        print(f"⚠️ El servidor rechazó el bloque. Código de error: {respuesta.status_code}")
        
except requests.exceptions.RequestException as e:
    print(f"❌ Falla de conexión al intentar enviar el bloque: {e}")

Iniciando envío en bloque de 25 registros al CRM...
✅ ¡Sincronización Exitosa! El bloque completo fue recibido.


Servidor Local

In [133]:
registros_list = df_master.to_dict(orient='records')

for r in registros_list:
    for c, v in r.items():
        if hasattr(v, 'isoformat'): r[c] = v.isoformat()
        elif pd.isna(v): r[c] = None

url_local = "http://127.0.0.1:8000/cargar_lote"

respuesta = requests.post(url_local, json=registros_list)

print(respuesta.status_code)
print(respuesta.json())

200
{'status': 'Éxito', 'Upsert_realizado': 25}


In [134]:
conexion = sqlite3.connect('external.db')

df_verificacion = pd.read_sql("SELECT * FROM catalogo_externo", conexion)

print(f"Filas encontradas en la DB local: {len(df_verificacion)}")
display(df_verificacion.head(10))

conexion.close()

Filas encontradas en la DB local: 25


,Producto,Precio,Calificacion_Promedio,Total_Resenas,Descripcion,Origen
0,Nokia 123,24.99,3,11,7 day battery,WebScraping_TouchPhones
1,LG Optimus,57.99,3,11,"3.2"" screen",WebScraping_TouchPhones
2,Samsung Galaxy,93.99,3,3,5 mpx. Android 5.0,WebScraping_TouchPhones
3,Nokia X,109.99,4,4,"Andoid, Jolla dualboot",WebScraping_TouchPhones
4,Sony Xperia,118.99,1,6,"GPS, waterproof",WebScraping_TouchPhones
5,Ubuntu Edge,499.99,1,2,Sapphire glass,WebScraping_TouchPhones
6,Iphone,899.99,1,10,White,WebScraping_TouchPhones
7,Iphone,899.99,2,8,Silver,WebScraping_TouchPhones
8,Iphone,899.99,1,1,Black,WebScraping_TouchPhones
9,iPhone 5s,199.99,3.7,3,"The iPhone 5s is a classic smartphone known for its compact design and advanced features during its release. While it's an older model, it still provides a reliable user experience.",API_Smartphones


In [140]:
df_master.iloc[[6]]

,Producto,Precio,Calificacion_Promedio,Total_Resenas,Descripcion,Origen
6,Iphone,899.99,1,10,White,WebScraping_TouchPhones


In [138]:
registros_list = df_master.iloc[[6]].to_dict(orient='records')

for r in registros_list:
    for c, v in r.items():
        if hasattr(v, 'isoformat'): r[c] = v.isoformat()
        elif pd.isna(v): r[c] = None

url_local = "http://127.0.0.1:8000/cargar_lote"

respuesta = requests.post(url_local, json=registros_list)

print(respuesta.status_code)
print(respuesta.json())

200
{'status': 'Éxito', 'Upsert_realizado': 1}


In [139]:
conexion = sqlite3.connect('external.db')

df_verificacion = pd.read_sql("SELECT * FROM catalogo_externo", conexion)

print(f"Filas encontradas en la DB local: {len(df_verificacion)}")
display(df_verificacion.head(10))

conexion.close()

Filas encontradas en la DB local: 23


,Producto,Precio,Calificacion_Promedio,Total_Resenas,Descripcion,Origen
0,Nokia 123,24.99,3,11,7 day battery,WebScraping_TouchPhones
1,LG Optimus,57.99,3,11,"3.2"" screen",WebScraping_TouchPhones
2,Samsung Galaxy,93.99,3,3,5 mpx. Android 5.0,WebScraping_TouchPhones
3,Nokia X,109.99,4,4,"Andoid, Jolla dualboot",WebScraping_TouchPhones
4,Sony Xperia,118.99,1,6,"GPS, waterproof",WebScraping_TouchPhones
5,Ubuntu Edge,499.99,1,2,Sapphire glass,WebScraping_TouchPhones
6,iPhone 5s,199.99,3.7,3,"The iPhone 5s is a classic smartphone known for its compact design and advanced features during its release. While it's an older model, it still provides a reliable user experience.",API_Smartphones
7,iPhone 6,299.99,4.0,3,"The iPhone 6 is a stylish and capable smartphone with a larger display and improved performance. It introduced new features and design elements, making it a popular choice in its time.",API_Smartphones
8,iPhone 13 Pro,1099.99,4.3,3,"The iPhone 13 Pro is a cutting-edge smartphone with a powerful camera system, high-performance chip, and stunning display. It offers advanced features for users who demand top-notch technology.",API_Smartphones
9,iPhone X,899.99,3.7,3,"The iPhone X is a flagship smartphone featuring a bezel-less OLED display, facial recognition technology (Face ID), and impressive performance. It represents a milestone in iPhone design and innovation.",API_Smartphones


In [142]:
df_master['ID'] = (
    df_master['Producto'].str.replace(r'[^a-zA-Z0-9]', '', regex=True) + 
    "_" + 
    df_master['Descripcion'].str.replace(r'[^a-zA-Z0-9]', '', regex=True).str[:10]
).str.upper()

In [143]:
df_master.head(10)

,Producto,Precio,Calificacion_Promedio,Total_Resenas,Descripcion,Origen,ID
0,Nokia 123,24.99,3,11,7 day battery,WebScraping_TouchPhones,NOKIA123_7DAYBATTER
1,LG Optimus,57.99,3,11,"3.2"" screen",WebScraping_TouchPhones,LGOPTIMUS_32SCREEN
2,Samsung Galaxy,93.99,3,3,5 mpx. Android 5.0,WebScraping_TouchPhones,SAMSUNGGALAXY_5MPXANDROI
3,Nokia X,109.99,4,4,"Andoid, Jolla dualboot",WebScraping_TouchPhones,NOKIAX_ANDOIDJOLL
4,Sony Xperia,118.99,1,6,"GPS, waterproof",WebScraping_TouchPhones,SONYXPERIA_GPSWATERPR
5,Ubuntu Edge,499.99,1,2,Sapphire glass,WebScraping_TouchPhones,UBUNTUEDGE_SAPPHIREGL
6,Iphone,899.99,1,10,White,WebScraping_TouchPhones,IPHONE_WHITE
7,Iphone,899.99,2,8,Silver,WebScraping_TouchPhones,IPHONE_SILVER
8,Iphone,899.99,1,1,Black,WebScraping_TouchPhones,IPHONE_BLACK
9,iPhone 5s,199.99,3.7,3,"The iPhone 5s is a classic smartphone known for its compact design and advanced features during its release. While it's an older model, it still provides a reliable user experience.",API_Smartphones,IPHONE5S_THEIPHONE5


In [144]:
registros_list = df_master.to_dict(orient='records')

for r in registros_list:
    for c, v in r.items():
        if hasattr(v, 'isoformat'): r[c] = v.isoformat()
        elif pd.isna(v): r[c] = None

url_local = "http://127.0.0.1:8000/cargar_lote"

respuesta = requests.post(url_local, json=registros_list)

print(respuesta.status_code)
print(respuesta.json())

200
{'status': 'Éxito', 'Upsert_realizado': 25}


In [145]:
conexion = sqlite3.connect('external.db')

df_verificacion = pd.read_sql("SELECT * FROM catalogo_externo", conexion)

print(f"Filas encontradas en la DB local: {len(df_verificacion)}")
display(df_verificacion.head(10))

conexion.close()

Filas encontradas en la DB local: 25


,Producto,Precio,Calificacion_Promedio,Total_Resenas,Descripcion,Origen,ID
0,Nokia 123,24.99,3,11,7 day battery,WebScraping_TouchPhones,NOKIA123_7DAYBATTER
1,LG Optimus,57.99,3,11,"3.2"" screen",WebScraping_TouchPhones,LGOPTIMUS_32SCREEN
2,Samsung Galaxy,93.99,3,3,5 mpx. Android 5.0,WebScraping_TouchPhones,SAMSUNGGALAXY_5MPXANDROI
3,Nokia X,109.99,4,4,"Andoid, Jolla dualboot",WebScraping_TouchPhones,NOKIAX_ANDOIDJOLL
4,Sony Xperia,118.99,1,6,"GPS, waterproof",WebScraping_TouchPhones,SONYXPERIA_GPSWATERPR
5,Ubuntu Edge,499.99,1,2,Sapphire glass,WebScraping_TouchPhones,UBUNTUEDGE_SAPPHIREGL
6,Iphone,899.99,1,10,White,WebScraping_TouchPhones,IPHONE_WHITE
7,Iphone,899.99,2,8,Silver,WebScraping_TouchPhones,IPHONE_SILVER
8,Iphone,899.99,1,1,Black,WebScraping_TouchPhones,IPHONE_BLACK
9,iPhone 5s,199.99,3.7,3,"The iPhone 5s is a classic smartphone known for its compact design and advanced features during its release. While it's an older model, it still provides a reliable user experience.",API_Smartphones,IPHONE5S_THEIPHONE5


In [146]:
df_master.iloc[[6]]

,Producto,Precio,Calificacion_Promedio,Total_Resenas,Descripcion,Origen,ID
6,Iphone,899.99,1,10,White,WebScraping_TouchPhones,IPHONE_WHITE


In [147]:
registros_list = df_master.iloc[[6]].to_dict(orient='records')

for r in registros_list:
    for c, v in r.items():
        if hasattr(v, 'isoformat'): r[c] = v.isoformat()
        elif pd.isna(v): r[c] = None

url_local = "http://127.0.0.1:8000/cargar_lote"

respuesta = requests.post(url_local, json=registros_list)

print(respuesta.status_code)
print(respuesta.json())

200
{'status': 'Éxito', 'Upsert_realizado': 1}


In [150]:
conexion = sqlite3.connect('external.db')

df_verificacion = pd.read_sql("SELECT * FROM catalogo_externo", conexion)

print(f"Filas encontradas en la DB local: {len(df_verificacion)}")
display(df_verificacion.head(30))

conexion.close()

Filas encontradas en la DB local: 25


,Producto,Precio,Calificacion_Promedio,Total_Resenas,Descripcion,Origen,ID
0,Nokia 123,24.99,3,11,7 day battery,WebScraping_TouchPhones,NOKIA123_7DAYBATTER
1,LG Optimus,57.99,3,11,"3.2"" screen",WebScraping_TouchPhones,LGOPTIMUS_32SCREEN
2,Samsung Galaxy,93.99,3,3,5 mpx. Android 5.0,WebScraping_TouchPhones,SAMSUNGGALAXY_5MPXANDROI
3,Nokia X,109.99,4,4,"Andoid, Jolla dualboot",WebScraping_TouchPhones,NOKIAX_ANDOIDJOLL
4,Sony Xperia,118.99,1,6,"GPS, waterproof",WebScraping_TouchPhones,SONYXPERIA_GPSWATERPR
5,Ubuntu Edge,499.99,1,2,Sapphire glass,WebScraping_TouchPhones,UBUNTUEDGE_SAPPHIREGL
6,Iphone,899.99,2,8,Silver,WebScraping_TouchPhones,IPHONE_SILVER
7,Iphone,899.99,1,1,Black,WebScraping_TouchPhones,IPHONE_BLACK
8,iPhone 5s,199.99,3.7,3,"The iPhone 5s is a classic smartphone known for its compact design and advanced features during its release. While it's an older model, it still provides a reliable user experience.",API_Smartphones,IPHONE5S_THEIPHONE5
9,iPhone 6,299.99,4.0,3,"The iPhone 6 is a stylish and capable smartphone with a larger display and improved performance. It introduced new features and design elements, making it a popular choice in its time.",API_Smartphones,IPHONE6_THEIPHONE6
